# Urdu Readability - Linear Regression Model
## Predicts avg_syllable/avg_word_length from linguistic features

In [ ]:
# Install dependencies (run once)
!pip install pandas scikit-learn openpyxl matplotlib joblib -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

# Load cleaned data (use CSV if preprocessed, else load and clean)
try:
    data = pd.read_csv("input_to_linear_regression2_cleaned.csv")
    print("Loaded cleaned CSV")
except FileNotFoundError:
    data = pd.read_excel("input_to_linear_regression2.xlsx")
    if "Unnamed: 8" in data.columns:
        data = data.drop(columns=["Unnamed: 8"])
    data = data.dropna()
    print("Loaded and cleaned from Excel")

print(f"Dataset shape: {data.shape}")

In [ ]:
# Feature columns (X) and target (y)
feature_cols = [
    "1-syllable_words", "2-syllable_words", "3-syllable_words", "4-syllable_words",
    "5-syllable_words", "6-syllable_words", "7-syllable_words", "8-syllable_words",
    "word_length-1", "word_length-2", "word_length-3", "word_length-4",
    "word_length-5", "word_length-6", "word_length-7", "word_length-8",
    "sentence_length"
]

X = data[feature_cols]
y = data["avg_syllable/avg_word_length"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Features:", feature_cols)
print("Target: avg_syllable/avg_word_length")

In [ ]:
# Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print("Model trained successfully.")

In [ ]:
# ALPHA, BETA, GAMMA values for Linear Regression
# y = alpha + beta_1*x_1 + beta_2*x_2 + ... + beta_n*x_n
# alpha = intercept, beta = coefficients, gamma = R² (model fit quality)

alpha = model.intercept_
beta = dict(zip(feature_cols, model.coef_))
gamma = model.score(X_test, y_test)  # R² score

print("="*60)
print("LINEAR REGRESSION PARAMETERS")
print("="*60)
print(f"\nALPHA (Intercept): {alpha:.6f}")
print(f"\nBETA (Coefficients):")
for feat, coef in beta.items():
    print(f"  {feat}: {coef:.6f}")
print(f"\nGAMMA (R² Score): {gamma:.6f}")
print(f"\nEquation: y = {alpha:.4f} + ", end="")
print(" + ".join([f"({coef:.4f})*{f}" for f, coef in list(beta.items())[:3]]) + " + ...")

In [ ]:
# Model evaluation
mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)

print("Model Evaluation:")
print(f"  R² (Train): {model.score(X_train, y_train):.4f}")
print(f"  R² (Test):  {gamma:.4f}")
print(f"  RMSE:       {rmse:.4f}")

In [ ]:
# Save the trained model (ready for deployment)
model_data = {
    "model": model,
    "alpha": alpha,
    "beta": beta,
    "gamma": gamma,
    "feature_cols": feature_cols,
}
joblib.dump(model_data, "urdu_readability_model.joblib")
print("Model saved to: urdu_readability_model.joblib")

# Load and use: model_data = joblib.load('urdu_readability_model.joblib')

# Fry Readability Diagram
Plots **syllables per 100 words** (Y-axis) vs **sentences per 100 words** (X-axis) — standard Fry graph format for readability assessment.

In [ ]:
# Compute Fry metrics for each sentence
# Fry Graph: X = syllables per 100 words, Y = sentences per 100 words
syll_cols = [f"{i}-syllable_words" for i in range(1, 9)]
data["total_syllables"] = sum(i * data[col] for i, col in enumerate(syll_cols, 1))
data["syllables_per_100_words"] = (data["total_syllables"] / data["sentence_length"]) * 100
data["sentences_per_100_words"] = 100 / data["sentence_length"]
plot_sample = data.sample(n=min(5000, len(data)), random_state=42)

In [ ]:
# Proper Fry Graph - curved grade boundaries, gradient, labeled regions
# (Matches standard Fry Graph format)

x_min, x_max = 108, 172
y_min, y_max = 2.0, 25.0

def fry_grade(syllables, sentences):
    s_norm = (syllables - 108) / 64
    p_norm = (sentences - 2) / 23
    difficulty = (1 - p_norm) * 0.6 + s_norm * 0.4 + 0.08 * (1 - p_norm) * s_norm
    return np.clip(1 + difficulty * 14, 1, 15)

xx = np.linspace(x_min, x_max, 200)
yy = np.linspace(y_min, y_max, 200)
XG, YG = np.meshgrid(xx, yy)
ZG = np.vectorize(fry_grade)(XG, YG)

fig, ax = plt.subplots(figsize=(14, 11))

# Gradient background
gradient = np.zeros((*ZG.shape, 4))
for i in range(ZG.shape[0]):
    for j in range(ZG.shape[1]):
        t = (i / ZG.shape[0] + j / ZG.shape[1]) / 2
        gradient[i, j] = (0.85 - t*0.3, 0.92 - t*0.2, 1.0, 0.4)
ax.imshow(gradient, extent=[x_min, x_max, y_min, y_max], origin='lower', aspect='auto', zorder=0)

# Grade regions (contour)
levels = np.arange(1, 17, 1)
ax.contourf(XG, YG, ZG, levels=levels, alpha=0.25, cmap='Blues', zorder=1)
ax.contour(XG, YG, ZG, levels=levels, colors='#2c5282', linewidths=1.2, alpha=0.8, zorder=2)

# Grade numbers in circles
for g in range(1, 16):
    s_frac, p_frac = (g - 1) / 15, 1 - (g - 1) / 15 * 0.9
    sx = x_min + (x_max - x_min) * (s_frac * 0.7 + 0.15)
    sy = y_min + (y_max - y_min) * (p_frac * 0.85 + 0.08)
    ax.add_patch(Circle((sx, sy), 0.8, color='white', ec='#1a365d', linewidth=1.5, zorder=3))
    ax.text(sx, sy, str(g), ha='center', va='center', fontsize=10, fontweight='bold', zorder=4)

# "long words" and "long sentences" regions
ax.add_patch(Rectangle((x_max-10, y_max-3), 10, 3, facecolor='#1a365d', alpha=0.75, zorder=1))
ax.text(x_max-5, y_max-1.5, 'long words', fontsize=8, ha='center', va='center', color='white', fontweight='bold', zorder=5)
ax.add_patch(Rectangle((x_min, y_min), 8, 2.5, facecolor='#1a365d', alpha=0.75, zorder=1))
ax.text(x_min+4, y_min+1.25, 'long sentences', fontsize=8, ha='center', va='center', color='white', fontweight='bold', zorder=5)

# Clean graph - NO data points (matches standard Fry Graph)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel("Average number of syllables per 100 words", fontsize=13, fontweight='bold')
ax.set_ylabel("Average number of sentences per 100 words", fontsize=13, fontweight='bold')
ax.set_title("Fry Graph\nFOR ESTIMATING READING AGES (GRADE LEVEL)", fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("fry_readability_diagram.png", dpi=150, bbox_inches="tight")
plt.show()
print("Fry diagram saved to: fry_readability_diagram.png")